# Advanced AI Lab 1 — Sentiment Analysis

This is the **single entry point** for all Lab 1 experiments.  
Run the cells top-to-bottom in each section you need.  
All heavy logic lives in the project files — this notebook just calls them.

| Section | Task |
|---|---|
| 0 — Setup | Check GPU, install dependencies, wandb login |
| 1 — Task 1.1 ANN | Simple ANN on small (1 K) and large (25 K) datasets |
| 2 — Task 1.1 BiLSTM | Bidirectional LSTM on small and large datasets |
| 3 — Task 1.2 BERT | Fine-tune BERT on the large Amazon dataset |
| 4 — Task 1.2 DistilBERT | Fine-tune DistilBERT on the large Amazon dataset |
| 5 — Grade 5 | BERT + DistilBERT on the public ~1 GB amazon_polarity dataset |
| 6 — Comparison | Side-by-side results across all models |
| 7 — W&B | View training curves at https://wandb.ai |

---
## Section 0 — Setup

Run this cell once before anything else.  
It verifies PyTorch, shows the GPU (if available), and confirms all modules load correctly.

In [1]:
import sys
import os

# Make sure the Lab 1 root is on the Python path so all imports resolve
project_root = os.path.dirname(os.path.abspath("__file__"))
sys.path.insert(0, project_root)

import torch
import config

print("=" * 55)
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
print(f"Device in use   : {config.DEVICE}")
print("=" * 55)
print()
print("Datasets found:")
print(f"  Small (1 K)  : {os.path.exists(config.SMALL_DATASET_PATH)}  →  {config.SMALL_DATASET_PATH}")
print(f"  Large (25 K) : {os.path.exists(config.LARGE_DATASET_PATH)}  →  {config.LARGE_DATASET_PATH}")
print()
print("All project modules are ready.")
print("Run any section below to start an experiment.")

PyTorch version : 2.11.0+cu130
CUDA available  : True
GPU             : NVIDIA GeForce RTX 2080 Ti
Device in use   : cuda

Datasets found:
  Small (1 K)  : True  →  /Labs/Lab1/amazon_cells_labelled.txt
  Large (25 K) : True  →  /Labs/Lab1/amazon_cells_labelled_LARGE_25K.txt

All project modules are ready.
Run any section below to start an experiment.


### Section 0.1 — Weights & Biases Login (run once)

All experiments log to **https://wandb.ai** automatically.  
Run the cell below to install wandb (if missing) and authenticate.  
Get your API key at **https://wandb.ai/authorize**

In [2]:
import subprocess, sys

import wandb
wandb.login()   # prompts for API key if not already stored

print()
print("Logged in to wandb.")
print("After running experiments, view all results at: https://wandb.ai")
print(f"Project name: {__import__('config').WANDB_PROJECT}")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: deeplab15group (deeplab15group-lule-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



Logged in to wandb.
After running experiments, view all results at: https://wandb.ai
Project name: advanced-ai-lab-1


---
## Section 1 — Task 1.1: Simple ANN

A feed-forward network trained on **TF-IDF bag-of-words features**.  
We run the same architecture twice — once on the 1 K small dataset, once on the 25 K large dataset — to observe how training-data volume impacts accuracy.

| Cell | Dataset | What we learn |
|---|---|---|
| 1.1 | Small (1 K) | Baseline — limited data |
| 1.2 | Large (25 K) | Effect of 25× more training data |

In [3]:
# 1.1  Simple ANN — small dataset (1 K reviews)
from experiments.task01_ann import main as run_ann

ann_small = run_ann(dataset="small")


Device: cuda
GPU   : NVIDIA GeForce RTX 2080 Ti

Loading ANN data  [small dataset] …
  Preprocessing text (classical) …
  Fitting TF-IDF on training split (will NOT see val / test) …
  TF-IDF vocab: 4,033 features
  Split: 700 train / 150 val / 150 test

Building ANN  (vocab_size=4,033) …
  [build_ann] dataset='small' → SmallANN (2 blocks, 128→64)
  Total parameters    : 524,994
  Trainable parameters: 524,994



══════════════════════════════════════════════════════════════
  Experiment  : Task01_ANN_Small
  Device      : cuda  (NVIDIA GeForce RTX 2080 Ti)
  Optimiser   : Adam  lr=0.0005
  Epochs      : 50
  Patience    : 5
  Trainable   : 524,994 parameters
  Split       : 70% train / 15% val / 15% test
  W&B project : advanced-ai-lab-1  (view at https://wandb.ai)
══════════════════════════════════════════════════════════════



  Epoch  1/50  |  train  loss=0.7144  acc=46.7%  |  val  loss=0.6925  acc=50.0%  f1=0.6667  prec=0.5000  rec=1.0000


  Epoch  2/50  |  train  loss=0.6750  acc=60.3%  |  val  loss=0.6767  acc=60.7%  f1=0.6093  prec=0.6053  rec=0.6133


  Epoch  3/50  |  train  loss=0.6306  acc=67.4%  |  val  loss=0.6423  acc=66.0%  f1=0.6331  prec=0.6875  rec=0.5867


...

  Epoch 14/50  |  train  loss=0.0319  acc=99.6%  |  val  loss=0.4892  acc=81.3%  f1=0.8205  prec=0.7901  rec=0.8533


  Epoch 15/50  |  train  loss=0.0386  acc=99.3%  |  val  loss=0.5054  acc=78.7%  f1=0.7975  prec=0.7590  rec=0.8400
  Early stopping triggered (no improvement for 5 epochs).



──────────────────────────────────────────────────────────────
  [FINAL]  Test Accuracy : 80.67%   F1 : 0.8027   Precision : 0.8194   Recall : 0.7867
──────────────────────────────────────────────────────────────



batch_loss,▇▇███▇▇▇▆▄▅▄▄▄▄▂▂▁▂▂▂▂▁▂▁▁▁▁▁▆▁▁▁▁▁▁▁▁▁▂
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_accuracy,▁▃▄▆▇██████████
train_loss,██▇▆▄▃▂▁▁▁▁▁▁▁▁
val_accuracy,▁▃▅▆▇███▇█████▇
val_f1,▃▁▂▅▇██▇▇███▇█▇
val_loss,██▆▄▂▁▁▂▃▃▃▃▄▂▂
val_precision,▁▃▅▇▇█▇▇▇████▇▇
val_recall,█▁▁▃▅▅▅▅▅▅▅▅▄▆▅
batch_loss,0.01334
epoch,15


Checkpoint saved → /Labs/Lab1/checkpoints/Task01_ANN_Small.pth
[Result] Experiment       : Task01_ANN_Small
[Result] Best Val Accuracy: 82.00%
[Result] Test Accuracy    : 80.67%
[Result] Test F1          : 0.8027



In [ ]:
# 1.2  Simple ANN — large dataset (25 K reviews)
from experiments.task01_ann import main as run_ann

ann_large = run_ann(dataset="large")

In [ ]:
# grade 5  Simple ANN — public dataset (~100 K reviews from amazon_polarity)
from experiments.task01_ann import main as run_ann

ann_public = run_ann(dataset="public")

---
## Section 2 — Task 1.1: Bidirectional LSTM

A BiLSTM with **learned word embeddings** that reads the sentence in both directions.  
Unlike the ANN, the BiLSTM preserves word order and captures long-range dependencies  
(e.g. negations, emphasis) — important signals for sentiment.

| Cell | Dataset | What we learn |
|---|---|---|
| 2.1 | Small (1 K) | Sequence model on limited data |
| 2.2 | Large (25 K) | Sequence model vs ANN on the same 25 K data |

In [ ]:
# 2.1  BiLSTM — small dataset (1 K reviews)
from experiments.task01_bilstm import main as run_bilstm

bilstm_small = run_bilstm(dataset="small")

In [ ]:
# 2.2  BiLSTM — large dataset (25 K reviews)
from experiments.task01_bilstm import main as run_bilstm

bilstm_large = run_bilstm(dataset="large")

In [ ]:
# grade 5  BiLSTM — public dataset (~100 K reviews from amazon_polarity)
from experiments.task01_bilstm import main as run_bilstm

bilstm_public = run_bilstm(dataset="public")

---
## Section 3 — Task 1.2: BERT Fine-Tuning

**BERT-base-uncased** was pre-trained on BookCorpus + English Wikipedia using  
masked language modelling and next-sentence prediction.  
We replace its classification head and fine-tune **all 12 transformer layers** for sentiment.

> ⚠ GPU strongly recommended — BERT has 110 M parameters.  
> On CPU, reduce `"epochs"` in `config.py → BERT_CONFIG` to `1`.

In [ ]:
# 3.1  BERT — fine-tuned on the large Amazon dataset (25 K reviews)
from experiments.task02_bert import main as run_bert

bert_results = run_bert()

---
## Section 4 — Task 1.2: DistilBERT Fine-Tuning

**DistilBERT-base-uncased** is a knowledge-distilled version of BERT:  
40% fewer parameters, 60% faster, ~97% of BERT's accuracy.  

Running both side-by-side provides a direct **complexity vs accuracy trade-off** study.

In [ ]:
# 4.1  DistilBERT — fine-tuned on the large Amazon dataset (25 K reviews)
from experiments.task02_distilbert import main as run_distilbert

distilbert_results = run_distilbert()

---
## Section 5 — Grade 5: Two Transformers on the Public Dataset (~1 GB)

Runs **BERT-base-uncased** and **DistilBERT-base-uncased** back-to-back,  
both fine-tuned on `amazon_polarity` from Hugging Face.

- **amazon_polarity** — ~3.6 M product reviews, ~1 GB download  
- Training is capped at `PUBLIC_MAX_SAMPLES` rows (set in `config.py`, default 100 K)  
- Both models train on the identical data split for a fair comparison

> ⚠ GPU strongly recommended.  On CPU, set `"epochs": 1` in
> `BERT_PUBLIC_CONFIG` and `DISTILBERT_PUBLIC_CONFIG` in `config.py`.

In [ ]:

# 5.1  BERT + DistilBERT on the public ~1 GB amazon_polarity dataset
from experiments.grade5_transformers_public import main as run_grade5

grade5_results = run_grade5()


---
## Section 6 — Task 1.3: Standardized Comparison (Required by Spec)

This section satisfies the Task 1.3 requirement:
> *"Use the **same test dataset** for Simple ANN, LSTM-based model and the Transformer."*

The cell below trains all three models on the **25 K large dataset**, evaluates them on the **exact same 15 % test split**, and prints:

- Test accuracy and F1 score for each model
- Parameter count (model complexity)
- Relative inference speed (compared to the ANN baseline)
- A written discussion of all three Task 1.3 comparison questions

The standardization is guaranteed by `RANDOM_SEED = 42` in `config.py` — every model's data loader applies the same stratified split to the same raw texts.

> ⚠ This cell trains all three models sequentially. Run Sections 1–5 first if you want individual results, or run this cell alone for the complete standardized comparison.

In [ ]:
# Task 1.3 — Standardized comparison: Simple ANN vs BiLSTM vs BERT on the identical test split
from experiments.task03_comparison import main as run_task03_comparison
comparison_results = run_task03_comparison()

---
## Section 6 — Model Comparison

Print a side-by-side table of all experiment results.  
Requires the cells above to have been run in the same session.

In [ ]:
# 6.1  Print side-by-side comparison of all models
results_dict = {}

try: results_dict["ANN  (small 1K)"]    = ann_small
except NameError: pass
try: results_dict["ANN  (large 25K)"]   = ann_large
except NameError: pass
try: results_dict["ANN  (public 100K)"] = ann_public
except NameError: pass
try: results_dict["BiLSTM (small 1K)"]  = bilstm_small
except NameError: pass
try: results_dict["BiLSTM (large 25K)"] = bilstm_large
except NameError: pass
try: results_dict["BiLSTM (public)"]    = bilstm_public
except NameError: pass
try: results_dict["BERT   (large 25K)"] = bert_results
except NameError: pass
try: results_dict["DistilBERT (25K)"]   = distilbert_results
except NameError: pass
try: results_dict["BERT   (public)"]    = grade5_results.get("BERT", {})
except (NameError, AttributeError): pass
try: results_dict["DistilBERT (pub)"]   = grade5_results.get("DistilBERT", {})
except (NameError, AttributeError): pass

print(f"{'Model':<25} {'Test Acc':>10}  {'Test F1':>10}  {'Best Val Acc':>12}")
print("-" * 62)
for name, res in results_dict.items():
    print(
        f"  {name:<23}"
        f"  {res.get('test_accuracy', float('nan')):>8.2f}%"
        f"  {res.get('test_f1', float('nan')):>10.4f}"
        f"  {res.get('best_val_accuracy', float('nan')):>10.2f}%"
    )
print("-" * 62)
print("\nView training curves at https://wandb.ai  (project: advanced-ai-lab-1)")

---
## Section 7 — Weights & Biases: View All Results

All experiments automatically stream metrics to **https://wandb.ai** — no local server needed.

### Viewing results
Open **https://wandb.ai** in any browser — all runs are grouped under project **`advanced-ai-lab-1`**.  
Use the **Charts** panel to compare training curves, val F1, and test accuracy side-by-side.

### Key comparisons to make
- **ANN small vs large** — impact of dataset size on bag-of-words features  
- **ANN vs BiLSTM** — bag-of-words vs sequence model on the same data  
- **BiLSTM vs BERT** — trained-from-scratch vs pre-trained transformer  
- **BERT vs DistilBERT** — accuracy trade-off for model compression  

> To rename the project, change `WANDB_PROJECT` in `config.py`.